# Randomized Search — a better use of the same budget

Grid search has a flaw that only shows up once you have more than two or three
hyperparameters: it spends its budget evenly, including on parameters that turn out not to
matter.

The validation curve at the end of the last notebook made this visible. Accuracy climbed
steeply as `C` came out of the over-regularised region and then went **flat**. Every grid
point in that flat region cost a full round of cross-validation and told us nothing new.

Now imagine four hyperparameters where two matter and two do not. A 5×5×5×5 grid is 625
combinations, and it evaluates each of the 25 useful settings of the two important
parameters exactly 25 times, once for every irrelevant combination of the others.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RSEED = 42

penguins = pd.read_csv("data/penguins.csv").dropna(subset=["sex"])

X = penguins.drop(columns="sex")
y = penguins["sex"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RSEED, stratify=y
)

numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
categorical = make_pipeline(SimpleImputer(strategy="most_frequent"),
                            OneHotEncoder(handle_unknown="ignore"))

preprocessor = make_column_transformer(
    (numeric, make_column_selector(dtype_include="number")),
    (categorical, make_column_selector(dtype_exclude="number")),
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

print(f"{len(X_train)} training penguins, {len(X_test)} held out")

## Sampling instead of enumerating

`RandomizedSearchCV` takes **distributions** rather than lists, and a budget: `n_iter` says
how many combinations to draw. Every draw gets a fresh value for *every* parameter, so a
budget of 30 tries 30 distinct values of each one — not five values seen six times each.

In [ ]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "classifier__C": loguniform(1e-3, 1e3),
    "preprocessor__pipeline-1__simpleimputer__strategy": ["mean", "median", "most_frequent"],
}

search = RandomizedSearchCV(
    model,
    param_distributions=param_distributions,
    n_iter=30,
    cv=5,
    scoring="accuracy",
    random_state=RSEED,
    n_jobs=-1,
)
search.fit(X_train, y_train)

print(f"best parameters : {search.best_params_}")
print(f"best cv score   : {search.best_score_:.4f}")
print(f"test accuracy   : {search.score(X_test, y_test):.4f}")

## Choosing the distribution

The distribution matters as much as the range. Two you will use constantly:

- **`loguniform(low, high)`** — for anything that spans orders of magnitude: `C`, `alpha`,
  `learning_rate`. It samples 0.001–0.01 as often as 100–1000, which is what you want when
  the interesting behaviour is on a log scale. A plain `uniform(0.001, 1000)` would put
  virtually every draw above 1 and never explore the small end at all.
- **`randint(low, high)`** — for integer counts: `n_estimators`, `max_depth`, `n_neighbors`.

You can still pass a plain list for genuinely categorical choices, as with the imputer above.

Look at what was actually sampled:

In [ ]:
results = pd.DataFrame(search.cv_results_)
sampled_C = results["param_classifier__C"].astype(float)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(np.log10(sampled_C), bins=15, color="#F5821F", edgecolor="white")
axes[0].set_xlabel("log10(C)")
axes[0].set_ylabel("draws")
axes[0].set_title("loguniform spreads evenly across the exponent")

axes[1].scatter(sampled_C, results["mean_test_score"], alpha=0.7, color="#2d6a9f")
axes[1].set_xscale("log")
axes[1].set_xlabel("C")
axes[1].set_ylabel("cv accuracy")
axes[1].set_title("30 distinct values of C, not 5")
plt.tight_layout()
plt.show()

The right-hand panel is the argument for random search in one picture. Thirty different
values of `C`, each tried once, tracing the shape of the curve — where a grid of the same
cost would have given six values tried five times.

## Setting `n_iter` deliberately

**`n_iter` defaults to 10, which is almost always too few.** It is the one parameter of the
search itself, and it should be a decision, not an accident.

The useful way to think about it: `n_iter` is your compute budget. Unlike a grid, you can set
it to whatever you can afford and the search adapts — doubling it explores twice as much
rather than forcing you to redesign the space.

In [ ]:
# A single run tells you little — the answer depends on which values happened to be drawn.
# Repeat each budget with eight different seeds and look at the spread.
for n_iter in [5, 10, 30, 60]:
    best = [
        RandomizedSearchCV(model, param_distributions, n_iter=n_iter, cv=5,
                           random_state=seed, n_jobs=-1)
        .fit(X_train, y_train).best_score_
        for seed in range(8)
    ]
    print(f"n_iter={n_iter:<4} mean {np.mean(best):.4f}   worst {min(best):.4f}   std {np.std(best):.4f}")

Read the **worst** and **std** columns, not the mean.

| `n_iter` | mean | worst | std |
|---|---|---|---|
| 5 | 0.9151 | 0.8876 | 0.0112 |
| 10 | 0.9221 | 0.9156 | 0.0044 |
| 30 | 0.9256 | 0.9236 | 0.0020 |
| 60 | 0.9261 | 0.9236 | 0.0019 |

With a budget of 5 the search is **unreliable**: depending on the draw it lands anywhere from
0.888 to a little over 0.92. The mean looks almost fine, which is exactly the trap — you only
run the search once, and you have no way of knowing whether you got a good draw.

By 30 the spread has collapsed to 0.002 and the answer is stable. Going to 60 buys essentially
nothing more.

**That is what `n_iter` controls: not the quality of the best possible answer, but how
reliably you find it.** The default of 10 sits right in the unreliable region, which is why
it should always be set deliberately.

Here the curve flattens early because there are only two parameters and one of them barely
matters. On a larger space it keeps climbing much longer — so scale `n_iter` with the size of
the space, not with habit.

## Grid or random?

| | `GridSearchCV` | `RandomizedSearchCV` |
|---|---|---|
| you specify | exact values | distributions + a budget |
| cost | product of all list lengths | exactly `n_iter` |
| adding a parameter | multiplies the cost | costs nothing extra |
| reproducible | inherently | with `random_state` |
| best for | few parameters, values you actually care about | many parameters, wide or continuous ranges |

The practical answer is usually **both, in sequence**: a broad random search to find the
region that matters, then a small grid inside that region to refine it.

> **Next:** we have now tuned twice, and both times reported `best_score_` alongside a test
> score. There is a subtler problem with that — one that needs cross-validation *around* the
> tuning, not just inside it.